# 319 Phase 2 — XGBoost Final Pool 외부 검증 (chan_browser n=324)

**참조**: `services/ai/train/model_experiment/xgboost_gyeom_final_pool.ipynb` (Phase 1, 보존 read-only)

## 본 phase scope (단순화)

**(a) eval-only**: Phase 1 `model.joblib` 으로 chan_browser 324 trials 외부 검증.
chan 단독 ROC AUC + threshold 0.40/0.75 confusion matrix + recall@0.40 / precision@0.75 + allow_FN_op 산출.

진단 figure / ADR-017 update / 후속 input 작성 등 부수 산출은 §2 결과 보고 후 분석 chat 합의로 별도 결정 — 본 phase 폐기.

## Phase 1 reference (보존)

- model: `services/ai/train/model_joblib/xgboost_gyeom_final_pool/model.joblib`
- input_features: `mouse_jerk_mean`, `mouse_max_speed_px_per_ms`
- meta: `production_ready=false / do_not_load_to_production=true / 5 blocking_flags`
- §5b lv3_kde recall@0.40 = 0.00 / allow_FN_op = 0.92 FAIL — chan 결과와 직접 비교 reference

## 평가 set

`services/ai/data/behavior_chan/behavior/trial_*.json` 324 trials (label balanced — macro 164 / human 160).
trial_loader: `list_trials_by_group("chan_browser")`.

## 산출물

본 phase 산출물 = 본 노트북 cells (§0 + §1 + §2). meta.json / model.joblib 신규 생성 X (Phase 1 4 artifacts 그대로 사용).

## Decision lock 요약 (plan)

- #1: 위치 = `behavior_chan/behavior/`, range 1~324, group `chan_browser`
- #2: (a) eval-only 채택
- #3: 신규 노트북 (본 파일)
- #4: §2 ROC AUC + cm + allow_FN_op (§3 진단 figure 폐기)
- #5: 좌표계 정합 — Round 4 §11 분포 추정 reference 보존, ADR update markdown 산출 폐기

## §0 Setup — Phase 1 reference 로드

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

SEED = 42
MODEL_NAME = "xgboost_gyeom_final_pool"
LABEL_MAPPING = {"human": 0, "macro": 1}

ROOT = Path.cwd()
if ROOT.name != "model_experiment":
    raise Exception(f"작업 디렉토리는 model_experiment 여야 함 (현재: {ROOT.name})")

# Phase 1 4 artifacts 경로
PHASE1_DIR = Path("../model_joblib") / MODEL_NAME
MODEL_PATH = PHASE1_DIR / "model.joblib"
META_PATH = PHASE1_DIR / "meta.json"
METRICS_PATH = PHASE1_DIR / "metrics.json"
INPUT_FEATURES_PATH = PHASE1_DIR / "input_features.json"

# chan_browser 그룹 (services/ai/train/EDA/trial_loader.py)
sys.path.insert(0, str(Path("../EDA").resolve()))
from trial_loader import list_trials_by_group, CHAN_BROWSER_RANGE

# input_features 로드 (Phase 1 final pool 2)
with INPUT_FEATURES_PATH.open("r", encoding="utf-8") as f:
    INPUT_FEATURES = json.load(f)

print("ROOT:", ROOT)
print("PHASE1_DIR:", PHASE1_DIR)
print("MODEL_PATH exists:", MODEL_PATH.exists())
print("CHAN_BROWSER_RANGE:", CHAN_BROWSER_RANGE)
print("INPUT_FEATURES (Phase 1):", INPUT_FEATURES)

ROOT: C:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_experiment
PHASE1_DIR: ..\model_joblib\xgboost_gyeom_final_pool
MODEL_PATH exists: True
CHAN_BROWSER_RANGE: (1, 324)
INPUT_FEATURES (Phase 1): ['mouse_jerk_mean', 'mouse_max_speed_px_per_ms']


## §1 chan data + dt sanity (결정 #5 prereq)

`list_trials_by_group("chan_browser")` → chan 324 trials 로드. final pool 2 feature (`mouse_jerk_mean`, `mouse_max_speed_px_per_ms`) + label 추출.

dt 비교:
- chan eventRows mouse_move event interval (`relative_ms` diff, `event_type == "mousemove"` 만 필터)
- 학습 풀 reference: `behavior/trial_900001.json` mouse_move dt
- 정합 OK → §2 진행 / 불일치 → 즉시 정지·보고 (memory: feedback_missing_data_policy 동일 정책)

In [2]:
# §1 chan data + dt sanity (1 cell 통합)

# 1. chan 324 trials 로드 + final pool 2 feature 추출
chan_metas = list_trials_by_group("chan_browser")
print(f"chan_browser trials: {len(chan_metas)}")

chan_rows = []
for meta in chan_metas:
    trial = json.loads(meta["path"].read_text(encoding="utf-8"))
    summary = trial.get("summary") or {}
    metrics = summary.get("metrics") or trial.get("metrics") or {}
    label = trial.get("label") or summary.get("label")
    if label not in LABEL_MAPPING:
        raise Exception(f"chan trial {meta['trial_id']} label invalid: {label!r}")
    row = {"trial_id": meta["trial_id"], "label": label, "label_int": LABEL_MAPPING[label]}
    for f in INPUT_FEATURES:
        v = metrics.get(f)
        if v is None:
            raise Exception(f"chan trial {meta['trial_id']} feature {f!r} missing (memory: feedback_missing_data_policy)")
        row[f] = v
    chan_rows.append(row)

chan_df = pd.DataFrame(chan_rows)
print(f"chan_df shape: {chan_df.shape}  columns: {list(chan_df.columns)}")
print(f"label balance: macro={(chan_df['label']=='macro').sum()}  human={(chan_df['label']=='human').sum()}")
print()

# 2. dt 비교 — chan vs pool trial_900001
def mousemove_dt(events, ts_field, type_field, type_value):
    move_ts = sorted([ev[ts_field] for ev in events
                      if ev.get(type_field) == type_value and ev.get(ts_field) is not None])
    return [move_ts[i+1] - move_ts[i] for i in range(len(move_ts) - 1)] if len(move_ts) >= 2 else []

def quantiles(vals, qs=(1, 50, 99)):
    if not vals: return [None] * len(qs)
    s = sorted(vals); n = len(s)
    return [s[max(0, min(n-1, int(n * q / 100)))] for q in qs]

chan_trial = json.loads(chan_metas[0]["path"].read_text(encoding="utf-8"))
chan_dt = mousemove_dt(chan_trial.get("eventRows") or [], "relative_ms", "event_type", "mousemove")
chan_q = quantiles(chan_dt)

pool_path = Path("../../data/behavior") / "trial_900001.json"
pool_trial = json.loads(pool_path.read_text(encoding="utf-8"))
pool_dt = mousemove_dt(pool_trial.get("eventRows") or [], "ts_ms", "event", "mouse_move")
pool_q = quantiles(pool_dt)

print(f"chan trial_00001 mouse_move dt (n={len(chan_dt)}):  p1={chan_q[0]} p50={chan_q[1]} p99={chan_q[2]}")
print(f"pool trial_900001 mouse_move dt (n={len(pool_dt)}): p1={pool_q[0]} p50={pool_q[1]} p99={pool_q[2]}")
print()

# 3. 정합 분기
chan_p50 = chan_q[1]; pool_p50 = pool_q[1]
if chan_p50 is None or pool_p50 is None:
    raise Exception("dt p50 산출 실패 (정지·보고)")
ratio = chan_p50 / pool_p50
print(f"chan p50 / pool p50 = {ratio:.3f}")
if not (0.1 <= ratio <= 10.0):
    raise Exception(
        f"dt p50 ratio {ratio:.3f} out of [0.1, 10.0] — chan vs pool 단위/scale 불일치.\n"
        f"chan p50={chan_p50} / pool p50={pool_p50}.\n"
        f"즉시 정지·보고 (memory: feedback_missing_data_policy)."
    )
print(f"\n[§1 PASS] dt 정합 OK (ratio {ratio:.3f} ∈ [0.1, 10.0]). §2 진행 가능.")

chan_browser trials: 324


chan_df shape: (324, 5)  columns: ['trial_id', 'label', 'label_int', 'mouse_jerk_mean', 'mouse_max_speed_px_per_ms']
label balance: macro=164  human=160

chan trial_00001 mouse_move dt (n=19):  p1=8 p50=21 p99=239
pool trial_900001 mouse_move dt (n=299): p1=27.67200000000048 p50=34.56999999999971 p99=684.4359999999997

chan p50 / pool p50 = 0.607

[§1 PASS] dt 정합 OK (ratio 0.607 ∈ [0.1, 10.0]). §2 진행 가능.


## §2 Phase 1 모델 평가 (chan 324 단독)

`joblib.load(MODEL_PATH)` → Phase 1 model. chan_df 의 final pool 2 feature 로 P(macro) 산출.

**Phase 1 정합 cm formula** (`xgboost_gyeom_final_pool.ipynb` cell 9 정의 동일):
- `block_fp_rate(t_block=0.75)` = P(macro) ≥ 0.75 인 human 비율
- `allow_fn_rate(t_allow)` = P(macro) ≤ t_allow 인 macro 비율 — `t_allow_design=0.40` / `t_allow_op=0.001`
- `review_rate(t_allow, t_block=0.75)` = 0.40~0.75 또는 0.001~0.75 between 비율
- `recall@0.40` = `recall_score(y, (proba > 0.40))` (strict `>`)
- `precision@0.75` = `precision_score(y, (proba >= 0.75))`

**단위:** trial-level (chan 324 trials, 1 trial = 1 row). window-level X.
**분모:** macro=164 / human=160 / total=324.
**Phase 1 §5b lv3_kde reference:** allow_FN_op=0.92 / allow_FN_design=1.0 / recall@0.40=0.0 / review_op=0.08.

In [3]:
# §2 Phase 1 모델 평가 (chan 324 단독)
from sklearn.metrics import roc_auc_score, precision_score, recall_score

# 1. Phase 1 model 로드
model = joblib.load(MODEL_PATH)
print(f"model loaded: {type(model).__name__}")

# 2. X / y 구성 (input_features.json 순서)
X = chan_df[INPUT_FEATURES].values
y = chan_df["label_int"].values
print(f"X shape: {X.shape}  y shape: {y.shape}  class counts: {dict(zip(*np.unique(y, return_counts=True)))}")

# NaN sanity (memory: feedback_missing_data_policy)
if np.isnan(X).any():
    raise Exception("X NaN 발견 — imputation·drop 금지, 즉시 정지·보고")

# 3. predict_proba → P(macro)
proba = model.predict_proba(X)[:, 1]
print(f"P(macro) range: [{proba.min():.4f}, {proba.max():.4f}]  mean={proba.mean():.4f}  median={np.median(proba):.4f}")

# 4. ROC AUC
auc = roc_auc_score(y, proba)
print(f"\n[chan ROC AUC] = {auc:.4f}")

# 5. Phase 1 정합 cm formula (xgboost_gyeom_final_pool.ipynb cell 9 정의 동일)
def block_fp_rate(y_true, y_proba, t_block=0.75):
    is_human = (y_true == 0)
    return float(((y_proba >= t_block) & is_human).sum() / is_human.sum()) if is_human.sum() else float("nan")

def allow_fn_rate(y_true, y_proba, t_allow):
    is_macro = (y_true == 1)
    return float(((y_proba <= t_allow) & is_macro).sum() / is_macro.sum()) if is_macro.sum() else float("nan")

def review_rate(y_true, y_proba, t_allow, t_block=0.75):
    return float(((y_proba > t_allow) & (y_proba < t_block)).sum() / len(y_proba))

T_BLOCK, T_ALLOW_DESIGN, T_ALLOW_OP = 0.75, 0.40, 0.001

# 6. Phase 1 5 secondary + 2 tertiary
m_block_fp = block_fp_rate(y, proba, T_BLOCK)
m_allow_fn_design = allow_fn_rate(y, proba, T_ALLOW_DESIGN)
m_allow_fn_op = allow_fn_rate(y, proba, T_ALLOW_OP)
m_review_design = review_rate(y, proba, T_ALLOW_DESIGN, T_BLOCK)
m_review_op = review_rate(y, proba, T_ALLOW_OP, T_BLOCK)

pred_block = (proba >= T_BLOCK).astype(int)
pred_recall = (proba > T_ALLOW_DESIGN).astype(int)
precision_at_075 = precision_score(y, pred_block, zero_division=0)
recall_at_040 = recall_score(y, pred_recall, zero_division=0)

# 7. cm counts (trial-level)
def cm_counts(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return tp, fp, tn, fn

tp_b, fp_b, tn_b, fn_b = cm_counts(y, pred_block)
tp_r, fp_r, tn_r, fn_r = cm_counts(y, pred_recall)

print(f"\n[T_block=0.75 (>= 0.75)]  cm: TP={tp_b} FP={fp_b} TN={tn_b} FN={fn_b}")
print(f"[T_design=0.40 (> 0.40)]  cm: TP={tp_r} FP={fp_r} TN={tn_r} FN={fn_r}")

# 8. 결과 요약 (chan trial-level vs Phase 1 §5b lv3_kde)
print("\n=== §2 결과 요약 (chan trial-level vs Phase 1 §5b lv3_kde) ===")
print(f"| metric                   | chan (n=324)         | §5b lv3_kde (n=50) |")
print(f"|---                       |---                   |---                  |")
print(f"| ROC AUC                  | {auc:.4f}              | (-)                 |")
print(f"| recall@0.40 (>0.40)      | {recall_at_040:.4f}              | 0.0000              |")
print(f"| precision@0.75 (>=0.75)  | {precision_at_075:.4f}              | (-)                 |")
print(f"| block_FP_rate (>=0.75)   | {m_block_fp:.4f}              | (-)                 |")
print(f"| allow_FN_design (<=0.40) | {m_allow_fn_design:.4f}              | 1.0000              |")
print(f"| allow_FN_op (<=0.001)    | {m_allow_fn_op:.4f}              | 0.9200              |")
print(f"| review_design (0.40-0.75)| {m_review_design:.4f}              | 0.0000              |")
print(f"| review_op (0.001-0.75)   | {m_review_op:.4f}              | 0.0800              |")

model loaded: Pipeline
X shape: (324, 2)  y shape: (324,)  class counts: {0: 160, 1: 164}
P(macro) range: [0.0000, 0.0000]  mean=0.0000  median=0.0000

[chan ROC AUC] = 0.5000

[T_block=0.75 (>= 0.75)]  cm: TP=0 FP=0 TN=160 FN=164
[T_design=0.40 (> 0.40)]  cm: TP=0 FP=0 TN=160 FN=164

=== §2 결과 요약 (chan trial-level vs Phase 1 §5b lv3_kde) ===
| metric                   | chan (n=324)         | §5b lv3_kde (n=50) |
|---                       |---                   |---                  |
| ROC AUC                  | 0.5000              | (-)                 |
| recall@0.40 (>0.40)      | 0.0000              | 0.0000              |
| precision@0.75 (>=0.75)  | 0.0000              | (-)                 |
| block_FP_rate (>=0.75)   | 0.0000              | (-)                 |
| allow_FN_design (<=0.40) | 1.0000              | 1.0000              |
| allow_FN_op (<=0.001)    | 1.0000              | 0.9200              |
| review_design (0.40-0.75)| 0.0000              | 0.0000             

C:\Users\SSAFY\miniforge3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
